<a href="https://colab.research.google.com/github/lillymitch/organized_archives/blob/main/data_processing_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import shutil
import os
from datasets import load_dataset, Image as HFImage
from PIL import Image
from io import BytesIO

wikiart_dataset = load_dataset(
    "huggan/wikiart",
    split="train",
    streaming=True
)

wikiart_dataset = wikiart_dataset.cast_column("image", HFImage(decode=False))
style_names = wikiart_dataset.features["style"].names

#defines path for small dataset to be created locally
small_dataset_path = "/content/wikiart_small"

#removes old dataset folder if it already exists, and creates new one
if os.path.exists(small_dataset_path):
    shutil.rmtree(small_dataset_path)
os.makedirs(small_dataset_path, exist_ok=True)

#classes to include
selected_classes = [
    "Impressionism",
    "Cubism",
    "Baroque",
    "Realism",
    "Expressionism",
]

#change later
max_per_class = 100

max_examples_to_check = 3000
examples_checked = 0
class_counts = {class_name: 0 for class_name in selected_classes}

#condense the dataset
for example in wikiart_dataset:

    examples_checked += 1
    if examples_checked >= max_examples_to_check:
        print("stopped early")
        break

    style_index = example["style"]
    style_name = style_names[style_index]

    #ONLY process if the style is in selected_classes and we haven't reached max_per_class for this style
    if style_name in selected_classes and class_counts[style_name] < max_per_class:

        class_folder = os.path.join(small_dataset_path, style_name)
        os.makedirs(class_folder, exist_ok=True)
        image_data = example["image"]

        #chatgpt try and except
        try:
            if image_data.get("bytes") is not None:
                image = Image.open(BytesIO(image_data["bytes"])).convert("RGB")
            else:
                image = Image.open(image_data["path"]).convert("RGB")
            save_path = os.path.join(
                class_folder,
                f"{class_counts[style_name]}.jpg"
            )

            image.save(save_path)
            class_counts[style_name] += 1
            print(class_counts)

        except:
            pass

    if all(class_counts[class_name] >= max_per_class for class_name in selected_classes):
        break

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

#ImageFolder dataset created from the condensed dataset
dataset = ImageFolder(
    root=small_dataset_path,
    transform=train_processor
)